# 🏛️ Zarya Digital Bank (Pakistan) — Week 10 Day 2
## QLoRA Supervised Fine-Tuning (SFT) with Unsloth, TRL & Weights & Biases

**Author:** Principal LLM & Fintech AI Engineer  
**Target Architecture:** Google Colab / Kaggle T4 GPU (16 GB VRAM)  
**Primary Candidate Model:** `Qwen/Qwen2.5-3B-Instruct`  
**Secondary Candidate Model:** `meta-llama/Llama-3.2-3B-Instruct`  
**Task:** Fine-tuning an UrduLish Banking Assistant with deterministic JSON output and SBP compliance.

### 1. Environment Setup & Dependency Installation
We install Unsloth (which provides custom Triton kernels for 2x to 5x faster training and 70% memory reduction), Hugging Face TRL, PEFT, bitsandbytes, and WandB.

In [ ]:
# Check GPU availability
!nvidia-smi

# Install Unsloth and modern fine-tuning libraries
!pip install --no-deps "xformers<0.0.28" "trl<0.10.0" peft accelerate bitsandbytes
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install wandb pydantic python-dotenv

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Memory Allocated: {torch.cuda.memory_allocated(0)/(1024**3):.2f} GB")

### 2. Candidate Model Selection & 4-bit NormalFloat Loading
We load `Qwen/Qwen2.5-3B-Instruct` in 4-bit precision using Unsloth's optimized `FastLanguageModel`.
Unsloth uses 4-bit NormalFloat (NF4) with Double Quantization to reduce model memory to under 2.0 GB.

In [ ]:
from unsloth import FastLanguageModel

max_seq_length = 2048
dtype = None  # None for auto-detection (Float16 on T4, Bfloat16 on Ampere/A100)
load_in_4bit = True

# Model ID for primary winning candidate
model_name = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

print(f"Model '{model_name}' successfully loaded in 4-bit NormalFloat precision!")

### 3. All-Linear LoRA Adapter Injection
We configure PEFT LoRA adapters across **all 7 linear projections** (`q_proj`, `k_proj`, `v_proj`, `o_proj`, `gate_proj`, `up_proj`, `down_proj`).
We set up our hyperparameter sweep configurations ($r=8, 16, 32$) with scaling factor $\alpha = 2r$.

In [ ]:
# Production Champion Configuration: Rank 16, Alpha 32
lora_rank = 16
lora_alpha = 32
lora_dropout = 0.05

model = FastLanguageModel.get_peft_model(
    model,
    r=lora_rank,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    bias="none",
    use_gradient_checkpointing="unsloth",  # True memory-efficient gradient checkpointing
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)

trainable_params, total_params = model.get_nb_trainable_parameters()
print(f"Trainable Parameters: {trainable_params:,} ({trainable_params/total_params*100:.3f}% of total {total_params:,})")

### 4. Dataset Ingestion & ChatML Formatting
We load `data/splits/train.jsonl` and `data/splits/val.jsonl`. Each record contains a conversational dialogue.
We map the dialogue into Qwen's native ChatML template:
`<|im_start|>system...<|im_end|><|im_start|>user...<|im_end|><|im_start|>assistant...<|im_end|>`

In [ ]:
import json
from datasets import Dataset

def load_jsonl_dataset(file_path):
    rows = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

# Adjust paths if running locally or in Google Colab
train_data = load_jsonl_dataset("data/splits/train.jsonl")
val_data = load_jsonl_dataset("data/splits/val.jsonl")

def format_chatml(conversation):
    formatted_text = ""
    for msg in conversation["messages"]:
        role = msg["role"]
        content = msg["content"]
        formatted_text += f"<|im_start|>{role}\n{content}<|im_end|>\n"
    return {"text": formatted_text}

train_dataset = Dataset.from_list([format_chatml(conv) for conv in train_data])
val_dataset = Dataset.from_list([format_chatml(conv) for conv in val_data])

print(f"Loaded {len(train_dataset):,} training instances and {len(val_dataset):,} validation instances.")
print("Sample ChatML string:\n", train_dataset[0]["text"][:350], "...")

### 5. Completion Masking (`train_on_responses_only`)
**Critical Guardrail Invariant:** We MUST NOT train the model on system instructions or customer queries.
We mask user prompt tokens with `labels = -100` so loss is computed strictly on assistant JSON completions.
This prevents prompt hallucination and ensures 100% focused learning on UrduLish response generation.

In [ ]:
import inspect
import torch
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only
from trl import SFTTrainer
from transformers import TrainingArguments

sig = inspect.signature(TrainingArguments.__init__)
eval_key = "eval_strategy" if "eval_strategy" in sig.parameters else "evaluation_strategy"

training_args = TrainingArguments(
    output_dir="results/checkpoints/qwen25_3b_sft",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,  # Effective batch size = 8
    warmup_steps=10,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=25,
    eval_steps=100,
    save_strategy="steps",
    save_steps=200,
    optim="paged_adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=42,
    report_to="none",  # Set to 'wandb' if you have WandB login configured
    **{eval_key: "steps"}
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,  # Can make training 5x faster for short sequences
    args=training_args,
)

# Apply Unsloth completion masking on ChatML assistant header
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

print("Trainer initialized with Completion Masking applied!")


### 6. Training Execution & Loss Monitoring
We run the training loop and monitor training vs validation loss curve.

In [ ]:
# Execute Fine-Tuning
trainer_stats = trainer.train()
print("Training complete!")
print(f"Final Train Loss: {trainer_stats.training_loss:.4f}")

### 7. Interactive UrduLish Verification & Schema Validation
We test our fine-tuned model against real-world Pakistani banking edge cases.

In [ ]:
FastLanguageModel.for_inference(model)  # Enables 2x faster inference

test_prompt = "<|im_start|>system\nYou are Zarya Assistant (Zarya Madadgaar), the official bilingual AI banking concierge for Zarya Digital Bank (Pakistan). Follow all SBP guidelines, reply in polite UrduLish, apologize empathetically for failed debits, never ask for OTP/PIN, and output strictly valid JSON conforming to ZaryaBankingAssistantResponse.<|im_end|>\n<|im_start|>user\nAoa! Maine abhi Raast se 15,000 transfer kiye thay bhai ko. Account se kat gaye par usay nahi milay. RRN: 421098765432. Foran check karein!<|im_end|>\n<|im_start|>assistant\n"

inputs = tokenizer([test_prompt], return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=512, use_cache=True, temperature=0.1)
response_text = tokenizer.batch_decode(outputs)[0]

print("Generated Output:\n")
print(response_text[len(test_prompt):])

### 8. Model Merging & GGUF Quantization for Local Fallback
We save the LoRA adapter and export a 4-bit GGUF model for local CPU inference in Ollama/llama.cpp.

In [ ]:
# 1. Save LoRA Adapters only (~60 MB)
model.save_pretrained("zarya-qwen25-3b-sft-lora")
tokenizer.save_pretrained("zarya-qwen25-3b-sft-lora")
print("LoRA adapter saved successfully!")

# 2. Save 4-bit GGUF for local Ollama / llama.cpp serving
# model.save_pretrained_gguf("zarya_qwen25_3b_gguf", tokenizer, quantization_method="q4_k_m")
# print("Exported Q4_K_M GGUF model for Ollama/llama.cpp!")